In [1]:
# ============================================================
# PROYECTO DE TESIS - CLAUDIA YINETH PINEDA CORTINA
#
# BLOQUE 0. CONFIGURACIÓN GENERAL DEL PROYECTO
#
# Título:
# Efectos de las amnistías tributarias en intereses del
# Impuesto Predial Unificado sobre la morosidad municipal
# en Colombia
#
# Disponibilidad de información CHIP: 2007-2025
# Periodo principal de análisis: 2012-2025
# Ventana previa para diagnósticos: 2007-2011
# ============================================================


# ------------------------------------------------------------
# 1. MONTAR GOOGLE DRIVE
# ------------------------------------------------------------

from google.colab import drive

drive.mount("/content/drive")


# ------------------------------------------------------------
# 2. INSTALAR LIBRERÍA PARA ARCHIVOS .XLSB
# ------------------------------------------------------------

!pip install -q pyxlsb


# ------------------------------------------------------------
# 3. IMPORTAR LIBRERÍAS
# ------------------------------------------------------------

import pandas as pd
import numpy as np

from pathlib import Path


# ------------------------------------------------------------
# 4. CONFIGURACIÓN DE VISUALIZACIÓN
# ------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


# ------------------------------------------------------------
# 5. DEFINIR CARPETA PRINCIPAL DEL PROYECTO
# ------------------------------------------------------------

RUTA_DATOS = Path(
    "/content/drive/MyDrive/Datos"
)


# ------------------------------------------------------------
# 6. DEFINIR PERIODOS DEL PROYECTO
# ------------------------------------------------------------

# Años disponibles en la base contable del CHIP
ANIO_INICIO_CHIP = 2007
ANIO_FIN_CHIP = 2025

# Periodo principal de análisis de las amnistías
ANIO_INICIO_ANALISIS = 2012
ANIO_FIN_ANALISIS = 2025

# Listas de años
ANIOS_CHIP = list(
    range(ANIO_INICIO_CHIP, ANIO_FIN_CHIP + 1)
)

ANIOS_PREVIOS = list(
    range(ANIO_INICIO_CHIP, ANIO_INICIO_ANALISIS)
)

ANIOS_ANALISIS = list(
    range(ANIO_INICIO_ANALISIS, ANIO_FIN_ANALISIS + 1)
)


# ------------------------------------------------------------
# 7. VERIFICAR CONFIGURACIÓN
# ------------------------------------------------------------

print("Proyecto:")
print("Tesis Claudia Yineth Pineda Cortina")

print("\nCarpeta principal de datos:")
print(RUTA_DATOS)

print("\n¿Existe la carpeta principal?")
print(RUTA_DATOS.exists())

print("\nDisponibilidad de información CHIP:")
print(f"{ANIO_INICIO_CHIP}-{ANIO_FIN_CHIP}")

print("\nVentana previa para diagnósticos:")
print(ANIOS_PREVIOS)

print("\nPeriodo principal de análisis:")
print(f"{ANIO_INICIO_ANALISIS}-{ANIO_FIN_ANALISIS}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Proyecto:
Tesis Claudia Yineth Pineda Cortina

Carpeta principal de datos:
/content/drive/MyDrive/Datos

¿Existe la carpeta principal?
True

Disponibilidad de información CHIP:
2007-2025

Ventana previa para diagnósticos:
[2007, 2008, 2009, 2010, 2011]

Periodo principal de análisis:
2012-2025


In [2]:
# ============================================================
# PROYECTO DE TESIS - CLAUDIA YINETH PINEDA CORTINA
#
# BLOQUE 1. CONSTRUCCIÓN DEL PANEL BASE DE MUNICIPIOS
# Periodo: 2007-2025
# ============================================================


# ------------------------------------------------------------
# 1. DEFINIR RUTA DE LA BASE DE MUNICIPIOS
# ------------------------------------------------------------

RUTA_ORIGINALES = RUTA_DATOS / "originales"

RUTA_MUNICIPIOS = RUTA_ORIGINALES / "municipios"

archivo_municipios = (
    RUTA_MUNICIPIOS / "DIVIPOLA_Municipios.xlsx"
)


# ------------------------------------------------------------
# 2. LEER BASE DE MUNICIPIOS
# ------------------------------------------------------------

municipios = pd.read_excel(
    archivo_municipios,
    sheet_name="Municipios",
    dtype={
        "codigo_departamento": str,
        "codigo_municipio": str
    }
)


# ------------------------------------------------------------
# 3. CONSERVAR ÚNICAMENTE LAS VARIABLES NECESARIAS
# ------------------------------------------------------------

municipios = municipios[
    [
        "codigo_departamento",
        "departamento",
        "codigo_municipio",
        "municipio"
    ]
].copy()


# ------------------------------------------------------------
# 4. ASEGURAR FORMATO CORRECTO DE LOS CÓDIGOS DIVIPOLA
# ------------------------------------------------------------

municipios["codigo_departamento"] = (
    municipios["codigo_departamento"]
    .str.strip()
    .str.zfill(2)
)

municipios["codigo_municipio"] = (
    municipios["codigo_municipio"]
    .str.strip()
    .str.zfill(5)
)


# ------------------------------------------------------------
# 5. ELIMINAR POSIBLES DUPLICADOS
# ------------------------------------------------------------

municipios = municipios.drop_duplicates(
    subset=[
        "codigo_departamento",
        "codigo_municipio"
    ]
).reset_index(drop=True)


# ------------------------------------------------------------
# 6. CREAR LOS AÑOS DEL PANEL
# ------------------------------------------------------------

# ANIOS_CHIP fue definido en el Bloque 0
# y contiene los años 2007-2025.

anios = pd.DataFrame({
    "año": ANIOS_CHIP
})


# ------------------------------------------------------------
# 7. CONSTRUIR PANEL MUNICIPIO × AÑO
# ------------------------------------------------------------

panel = municipios.merge(
    anios,
    how="cross"
)


# ------------------------------------------------------------
# 8. ORDENAR EL PANEL
#    Código de departamento → Código de municipio → Año
# ------------------------------------------------------------

panel = panel.sort_values(
    by=[
        "codigo_departamento",
        "codigo_municipio",
        "año"
    ]
).reset_index(drop=True)


# ------------------------------------------------------------
# 9. VERIFICAR RESULTADOS
# ------------------------------------------------------------

observaciones_esperadas = (
    len(municipios) * len(anios)
)

print("Número de municipios:", len(municipios))
print("Número de años:", len(anios))

print(
    "Años del panel:",
    anios["año"].min(),
    "-",
    anios["año"].max()
)

print(
    "Número total de observaciones:",
    len(panel)
)

print(
    "Número esperado de observaciones:",
    observaciones_esperadas
)

print(
    "¿El panel tiene el tamaño esperado?",
    len(panel) == observaciones_esperadas
)

display(panel.head(20))

Número de municipios: 1103
Número de años: 19
Años del panel: 2007 - 2025
Número total de observaciones: 20957
Número esperado de observaciones: 20957
¿El panel tiene el tamaño esperado? True


,codigo_departamento,departamento,codigo_municipio,municipio,año
0,05,ANTIOQUIA,05001,MEDELLÍN,2007
1,05,ANTIOQUIA,05001,MEDELLÍN,2008
2,05,ANTIOQUIA,05001,MEDELLÍN,2009
3,05,ANTIOQUIA,05001,MEDELLÍN,2010
4,05,ANTIOQUIA,05001,MEDELLÍN,2011
5,05,ANTIOQUIA,05001,MEDELLÍN,2012
6,05,ANTIOQUIA,05001,MEDELLÍN,2013
7,05,ANTIOQUIA,05001,MEDELLÍN,2014
8,05,ANTIOQUIA,05001,MEDELLÍN,2015
9,05,ANTIOQUIA,05001,MEDELLÍN,2016


In [3]:
# ============================================================
# PROYECTO DE TESIS - CLAUDIA YINETH PINEDA CORTINA
#
# BLOQUE 2. RECAUDO DEL IMPUESTO PREDIAL EN EL FUT
#
# R_ACT_FUT = Recaudo del Impuesto Predial Unificado
#             correspondiente a la vigencia actual
#
# R_ANT_FUT = Recaudo del Impuesto Predial Unificado
#             correspondiente a vigencias anteriores
#
# Fuente: FUT - Categoría Ingresos
# Periodo disponible: 2012-2020
#
# El panel base cubre 2007-2025.
# Para 2007-2011 y 2021-2025 las variables FUT
# quedarán como valores ausentes (NaN).
# ============================================================


# ------------------------------------------------------------
# 1. DEFINIR RUTA DE LAS BASES DEL IMPUESTO PREDIAL
# ------------------------------------------------------------

RUTA_PREDIAL = RUTA_ORIGINALES / "predial"


# ------------------------------------------------------------
# 2. DEFINIR LOS AÑOS DISPONIBLES EN EL FUT
# ------------------------------------------------------------

ANIOS_FUT = list(
    range(2012, 2021)
)


# ------------------------------------------------------------
# 3. FUNCIÓN AUXILIAR PARA CONVERTIR A NUMÉRICO
# ------------------------------------------------------------

def a_numerico(serie):
    """
    Convierte una columna a formato numérico.

    Si Excel ya la leyó como numérica, conserva sus valores.
    Los valores que no puedan convertirse quedan como NaN.
    """

    if pd.api.types.is_numeric_dtype(serie):

        return pd.to_numeric(
            serie,
            errors="coerce"
        )

    return pd.to_numeric(
        serie.astype(str)
             .str.strip()
             .str.replace("$", "", regex=False)
             .str.replace(" ", "", regex=False)
             .str.replace(",", "", regex=False),
        errors="coerce"
    )


# ------------------------------------------------------------
# 4. CÓDIGOS FUT QUE NECESITAMOS
# ------------------------------------------------------------

CODIGO_PREDIAL_ACTUAL = "TI.A.1.3.1"

CODIGO_PREDIAL_ANTERIOR = "TI.A.1.3.2"


# ------------------------------------------------------------
# 5. CREAR LISTA PARA ALMACENAR LOS RESULTADOS
# ------------------------------------------------------------

bases_fut = []


# ============================================================
# PARTE A. LEER FUT 2012-2020
# ============================================================

for año in ANIOS_FUT:

    print(f"\nLeyendo FUT {año}...")


    # --------------------------------------------------------
    # 6. DEFINIR NOMBRE DEL ARCHIVO
    # --------------------------------------------------------

    if año == 2012:

        archivo_fut = (
            RUTA_PREDIAL /
            "2012_FutIngresos.xlsx"
        )

    else:

        archivo_fut = (
            RUTA_PREDIAL /
            f"{año}_FUTIngresos.xlsx"
        )


    # --------------------------------------------------------
    # 7. VERIFICAR QUE EL ARCHIVO EXISTA
    # --------------------------------------------------------

    if not archivo_fut.exists():

        raise FileNotFoundError(
            f"No se encontró el archivo FUT {año}: "
            f"{archivo_fut}"
        )


    # --------------------------------------------------------
    # 8. LEER ARCHIVO FUT
    # --------------------------------------------------------

    df = pd.read_excel(
        archivo_fut,
        dtype={
            "Cód. DANE Municipio": str,
            "Código Concepto": str
        }
    )


    # --------------------------------------------------------
    # 9. VERIFICAR COLUMNAS NECESARIAS
    # --------------------------------------------------------

    columnas_necesarias = [
        "Cód. DANE Municipio",
        "Código Concepto",
        "Recaudo"
    ]

    columnas_faltantes = [
        columna
        for columna in columnas_necesarias
        if columna not in df.columns
    ]

    if columnas_faltantes:

        raise ValueError(
            f"FUT {año}: faltan las columnas "
            f"{columnas_faltantes}"
        )


    # --------------------------------------------------------
    # 10. LIMPIAR CÓDIGO DEL CONCEPTO
    # --------------------------------------------------------

    df["Código Concepto"] = (
        df["Código Concepto"]
        .astype(str)
        .str.strip()
    )


    # --------------------------------------------------------
    # 11. SELECCIONAR LOS DOS CONCEPTOS DEL PREDIAL
    #
    # TI.A.1.3.1 = Vigencia actual
    # TI.A.1.3.2 = Vigencias anteriores
    # --------------------------------------------------------

    df = df[
        df["Código Concepto"].isin(
            [
                CODIGO_PREDIAL_ACTUAL,
                CODIGO_PREDIAL_ANTERIOR
            ]
        )
    ].copy()


    # --------------------------------------------------------
    # 12. CONSTRUIR CÓDIGO MUNICIPAL
    # --------------------------------------------------------

    df["codigo_municipio"] = (
        df["Cód. DANE Municipio"]
        .astype(str)
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
        .str.zfill(5)
    )


    # --------------------------------------------------------
    # 13. CONVERTIR EL RECAUDO A NUMÉRICO
    # --------------------------------------------------------

    df["recaudo_fut"] = a_numerico(
        df["Recaudo"]
    )

    df["año"] = año


    # --------------------------------------------------------
    # 14. VERIFICAR LOS CONCEPTOS ENCONTRADOS
    # --------------------------------------------------------

    cantidad_actual = (
        df["Código Concepto"]
        .eq(CODIGO_PREDIAL_ACTUAL)
        .sum()
    )

    cantidad_anterior = (
        df["Código Concepto"]
        .eq(CODIGO_PREDIAL_ANTERIOR)
        .sum()
    )

    print(
        "  Registros de vigencia actual:",
        cantidad_actual
    )

    print(
        "  Registros de vigencias anteriores:",
        cantidad_anterior
    )


    # --------------------------------------------------------
    # 15. AGRUPAR POSIBLES REGISTROS REPETIDOS
    #
    # min_count=1 evita convertir en cero los casos donde
    # todos los valores estén ausentes.
    # --------------------------------------------------------

    df = (
        df.groupby(
            [
                "codigo_municipio",
                "año",
                "Código Concepto"
            ],
            as_index=False
        )
        .agg(
            recaudo_fut=(
                "recaudo_fut",
                lambda x: x.sum(min_count=1)
            )
        )
    )


    # --------------------------------------------------------
    # 16. PASAR LOS DOS CONCEPTOS A COLUMNAS
    # --------------------------------------------------------

    df = (
        df.pivot(
            index=[
                "codigo_municipio",
                "año"
            ],
            columns="Código Concepto",
            values="recaudo_fut"
        )
        .reset_index()
    )


    # --------------------------------------------------------
    # 17. GARANTIZAR QUE EXISTAN LAS DOS COLUMNAS
    # --------------------------------------------------------

    if CODIGO_PREDIAL_ACTUAL not in df.columns:

        df[CODIGO_PREDIAL_ACTUAL] = np.nan

    if CODIGO_PREDIAL_ANTERIOR not in df.columns:

        df[CODIGO_PREDIAL_ANTERIOR] = np.nan


    # --------------------------------------------------------
    # 18. CAMBIAR LOS CÓDIGOS POR NOMBRES DE VARIABLES
    #
    # R_ACT_FUT = Recaudo de la vigencia actual
    # R_ANT_FUT = Recaudo de vigencias anteriores
    # --------------------------------------------------------

    df = df.rename(
        columns={
            CODIGO_PREDIAL_ACTUAL: "R_ACT_FUT",
            CODIGO_PREDIAL_ANTERIOR: "R_ANT_FUT"
        }
    )


    # --------------------------------------------------------
    # 19. CONSERVAR SOLAMENTE LAS VARIABLES NECESARIAS
    # --------------------------------------------------------

    df = df[
        [
            "codigo_municipio",
            "año",
            "R_ACT_FUT",
            "R_ANT_FUT"
        ]
    ].copy()


    # --------------------------------------------------------
    # 20. VERIFICAR UNICIDAD MUNICIPIO-AÑO
    # --------------------------------------------------------

    duplicados = df.duplicated(
        subset=[
            "codigo_municipio",
            "año"
        ]
    ).sum()

    if duplicados > 0:

        raise ValueError(
            f"FUT {año}: se encontraron "
            f"{duplicados} registros duplicados "
            f"municipio-año."
        )


    # --------------------------------------------------------
    # 21. MOSTRAR DISPONIBILIDAD DEL AÑO
    # --------------------------------------------------------

    print(
        f"  Municipios con R_ACT_FUT: "
        f"{df['R_ACT_FUT'].notna().sum()}"
    )

    print(
        f"  Municipios con R_ANT_FUT: "
        f"{df['R_ANT_FUT'].notna().sum()}"
    )


    bases_fut.append(df)


# ============================================================
# PARTE B. UNIR FUT 2012-2020
# ============================================================

recaudo_fut = pd.concat(
    bases_fut,
    ignore_index=True
)


# ------------------------------------------------------------
# 22. VERIFICAR UNICIDAD EN LA BASE FUT COMPLETA
# ------------------------------------------------------------

duplicados_fut = recaudo_fut.duplicated(
    subset=[
        "codigo_municipio",
        "año"
    ]
).sum()

if duplicados_fut > 0:

    raise ValueError(
        "La base FUT consolidada tiene "
        f"{duplicados_fut} duplicados municipio-año."
    )


print("\n========================================")
print("FUT 2012-2020 TERMINADO")
print("========================================")

print(
    "Observaciones FUT:",
    len(recaudo_fut)
)

print(
    "Observaciones con R_ACT_FUT:",
    recaudo_fut["R_ACT_FUT"].notna().sum()
)

print(
    "Observaciones con R_ANT_FUT:",
    recaudo_fut["R_ANT_FUT"].notna().sum()
)


# ============================================================
# PARTE C. INCORPORAR LOS DOS RECAUDOS AL PANEL
# ============================================================

# Permite volver a ejecutar este bloque sin crear columnas
# duplicadas con terminaciones _x y _y.

panel = panel.drop(
    columns=[
        "R_ACT_FUT",
        "R_ANT_FUT"
    ],
    errors="ignore"
)

panel = panel.merge(
    recaudo_fut,
    on=[
        "codigo_municipio",
        "año"
    ],
    how="left",
    validate="one_to_one"
)


# ------------------------------------------------------------
# 23. ORDENAR NUEVAMENTE EL PANEL
# ------------------------------------------------------------

panel = panel.sort_values(
    by=[
        "codigo_departamento",
        "codigo_municipio",
        "año"
    ]
).reset_index(drop=True)


# ============================================================
# PARTE D. VERIFICAR RESULTADO FINAL
# ============================================================

print("\n========================================")
print("RESULTADO DEL BLOQUE 2")
print("========================================")

print(
    "Observaciones totales del panel:",
    len(panel)
)

print(
    "R_ACT_FUT disponibles:",
    panel["R_ACT_FUT"].notna().sum()
)

print(
    "R_ACT_FUT ausentes:",
    panel["R_ACT_FUT"].isna().sum()
)

print(
    "R_ANT_FUT disponibles:",
    panel["R_ANT_FUT"].notna().sum()
)

print(
    "R_ANT_FUT ausentes:",
    panel["R_ANT_FUT"].isna().sum()
)


# ------------------------------------------------------------
# 24. DISPONIBILIDAD POR AÑO
# ------------------------------------------------------------

disponibilidad_anual = (
    panel.groupby("año")
    .agg(
        municipios_panel=(
            "codigo_municipio",
            "size"
        ),
        R_ACT_FUT_disponibles=(
            "R_ACT_FUT",
            "count"
        ),
        R_ANT_FUT_disponibles=(
            "R_ANT_FUT",
            "count"
        )
    )
    .reset_index()
)

display(disponibilidad_anual)


# ------------------------------------------------------------
# 25. MOSTRAR UNA MUESTRA DEL PANEL
# ------------------------------------------------------------

display(
    panel.head(30)
)


Leyendo FUT 2012...
  Registros de vigencia actual: 1070
  Registros de vigencias anteriores: 833
  Municipios con R_ACT_FUT: 1070
  Municipios con R_ANT_FUT: 832

Leyendo FUT 2013...
  Registros de vigencia actual: 1083
  Registros de vigencias anteriores: 887
  Municipios con R_ACT_FUT: 1083
  Municipios con R_ANT_FUT: 886

Leyendo FUT 2014...
  Registros de vigencia actual: 1083
  Registros de vigencias anteriores: 908
  Municipios con R_ACT_FUT: 1083
  Municipios con R_ANT_FUT: 908

Leyendo FUT 2015...
  Registros de vigencia actual: 1026
  Registros de vigencias anteriores: 842
  Municipios con R_ACT_FUT: 1026
  Municipios con R_ANT_FUT: 842

Leyendo FUT 2016...
  Registros de vigencia actual: 1070
  Registros de vigencias anteriores: 879
  Municipios con R_ACT_FUT: 1070
  Municipios con R_ANT_FUT: 879

Leyendo FUT 2017...
  Registros de vigencia actual: 1077
  Registros de vigencias anteriores: 933
  Municipios con R_ACT_FUT: 1077
  Municipios con R_ANT_FUT: 933

Leyendo FUT 201

,año,municipios_panel,R_ACT_FUT_disponibles,R_ANT_FUT_disponibles
0,2007,1103,0,0
1,2008,1103,0,0
2,2009,1103,0,0
3,2010,1103,0,0
4,2011,1103,0,0
5,2012,1103,1069,832
6,2013,1103,1082,886
7,2014,1103,1082,908
8,2015,1103,1025,842
9,2016,1103,1069,879


,codigo_departamento,departamento,codigo_municipio,municipio,año,R_ACT_FUT,R_ANT_FUT
0,05,ANTIOQUIA,05001,MEDELLÍN,2007,NaN,NaN
1,05,ANTIOQUIA,05001,MEDELLÍN,2008,NaN,NaN
2,05,ANTIOQUIA,05001,MEDELLÍN,2009,NaN,NaN
3,05,ANTIOQUIA,05001,MEDELLÍN,2010,NaN,NaN
4,05,ANTIOQUIA,05001,MEDELLÍN,2011,NaN,NaN
5,05,ANTIOQUIA,05001,MEDELLÍN,2012,3.760985e+08,4.791869e+07
6,05,ANTIOQUIA,05001,MEDELLÍN,2013,4.576889e+08,8.247165e+07
7,05,ANTIOQUIA,05001,MEDELLÍN,2014,4.935976e+08,6.027679e+07
8,05,ANTIOQUIA,05001,MEDELLÍN,2015,5.226282e+08,7.616222e+07
9,05,ANTIOQUIA,05001,MEDELLÍN,2016,5.517421e+08,6.199670e+07


In [4]:
# ============================================================
# PROYECTO DE TESIS - CLAUDIA YINETH PINEDA CORTINA
#
# BLOQUE 3. CONSTRUCCIÓN DE L_CHIP, C_CHIP Y C_CIERRE_CHIP
#
# Fuente:
# Contaduría General de la Nación - CHIP
#
# Periodo utilizado: 2007-2025
# ============================================================


# ------------------------------------------------------------
# 1. IMPORTAR LIBRERÍA ADICIONAL
# ------------------------------------------------------------

import unicodedata


# ------------------------------------------------------------
# 2. DEFINIR RUTA DEL ARCHIVO CHIP
# ------------------------------------------------------------

RUTA_CHIP = RUTA_ORIGINALES / "chip"

archivo_chip = (
    RUTA_CHIP
    / "CHIP_CGN_2007_2026T2_EntGob_130507_131007_410507.xls"
)


# ------------------------------------------------------------
# 3. VERIFICAR QUE EL ARCHIVO EXISTA
# ------------------------------------------------------------

if not archivo_chip.exists():

    raise FileNotFoundError(
        "No se encontró el archivo CHIP en:\n"
        f"{archivo_chip}"
    )

print("Archivo CHIP:")
print(archivo_chip)


# ------------------------------------------------------------
# 4. LEER EL ARCHIVO CHIP
#
# Aunque termina en .xls, realmente es un archivo de texto
# UTF-16 separado por barras verticales.
# ------------------------------------------------------------

chip = pd.read_csv(
    archivo_chip,
    sep="|",
    encoding="utf-16",
    dtype=str,
    keep_default_na=False
)


# ------------------------------------------------------------
# 5. RENOMBRAR COLUMNAS
# ------------------------------------------------------------

chip = chip.rename(
    columns={
        "Fecha": "fecha",
        "Codigo_entidad": "codigo_entidad_chip",
        "Razon_social": "razon_social_chip",
        "Código": "cuenta",
        "Descripción": "descripcion",
        "Saldo_inicial": "saldo_inicial",
        "Movimientos_debito": "movimientos_debito",
        "Movimientos_credito": "movimientos_credito",
        "Saldo_final": "saldo_final",
        "Saldo_corriente": "saldo_corriente",
        "Saldo_no_corriente": "saldo_no_corriente"
    }
)


# ------------------------------------------------------------
# 6. CONVERTIR LA FECHA
# ------------------------------------------------------------

chip["fecha"] = pd.to_datetime(
    chip["fecha"],
    format="%Y-%m-%d",
    errors="coerce"
)

if chip["fecha"].isna().any():

    raise ValueError(
        "Se encontraron fechas CHIP que no pudieron convertirse."
    )

chip["año"] = chip["fecha"].dt.year
chip["trimestre"] = chip["fecha"].dt.quarter


# ------------------------------------------------------------
# 7. CONVERTIR LAS VARIABLES MONETARIAS
#
# Los decimales vienen separados por coma.
# No se realiza ninguna modificación de escala.
# ------------------------------------------------------------

columnas_monetarias = [
    "saldo_inicial",
    "movimientos_debito",
    "movimientos_credito",
    "saldo_final",
    "saldo_corriente",
    "saldo_no_corriente"
]

for columna in columnas_monetarias:

    chip[columna] = pd.to_numeric(
        chip[columna]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )


# ------------------------------------------------------------
# 8. CONSERVAR EL PERIODO DEL PANEL
#
# Se excluye 2026 porque solamente contiene dos trimestres
# y el panel termina en 2025.
# ------------------------------------------------------------

chip = chip[
    chip["año"].between(2007, 2025)
].copy()


# ------------------------------------------------------------
# 9. VERIFICAR LAS CUENTAS DESCARGADAS
# ------------------------------------------------------------

cuentas_esperadas = {
    "1.3.05.07",
    "1.3.10.07",
    "4.1.05.07"
}

cuentas_en_archivo = set(
    chip["cuenta"].dropna().unique()
)

cuentas_faltantes = (
    cuentas_esperadas - cuentas_en_archivo
)

if cuentas_faltantes:

    raise ValueError(
        "Faltan las siguientes cuentas CHIP:\n"
        f"{sorted(cuentas_faltantes)}"
    )

print("\nCuentas encontradas:")

for cuenta in sorted(cuentas_en_archivo):

    print(" ", cuenta)


# ============================================================
# IDENTIFICACIÓN DE LOS MUNICIPIOS
# ============================================================


# ------------------------------------------------------------
# 10. FUNCIÓN PARA NORMALIZAR NOMBRES
#
# Elimina tildes, signos y diferencias entre mayúsculas
# y minúsculas.
# ------------------------------------------------------------

def normalizar_texto(valor):

    if pd.isna(valor):
        return ""

    valor = str(valor)

    valor = unicodedata.normalize(
        "NFKD",
        valor
    )

    valor = (
        valor
        .encode("ascii", "ignore")
        .decode("ascii")
        .upper()
    )

    valor = "".join(
        caracter if caracter.isalnum() else " "
        for caracter in valor
    )

    return " ".join(valor.split())


# ------------------------------------------------------------
# 11. CREAR MAESTRO MUNICIPAL A PARTIR DEL PANEL
# ------------------------------------------------------------

maestro_municipios = (
    panel[
        [
            "codigo_departamento",
            "departamento",
            "codigo_municipio",
            "municipio"
        ]
    ]
    .drop_duplicates()
    .copy()
)

maestro_municipios["codigo_departamento"] = (
    maestro_municipios["codigo_departamento"]
    .astype(str)
    .str.zfill(2)
)

maestro_municipios["codigo_municipio"] = (
    maestro_municipios["codigo_municipio"]
    .astype(str)
    .str.zfill(5)
)

maestro_municipios["municipio_norm"] = (
    maestro_municipios["municipio"]
    .map(normalizar_texto)
)

maestro_municipios["departamento_norm"] = (
    maestro_municipios["departamento"]
    .map(normalizar_texto)
)


# ------------------------------------------------------------
# 12. CREAR TABLA ÚNICA DE ENTIDADES CHIP
# ------------------------------------------------------------

entidades_chip = (
    chip[
        [
            "codigo_entidad_chip",
            "razon_social_chip"
        ]
    ]
    .drop_duplicates()
    .copy()
)


# Eliminar el código que aparece entre paréntesis al final
# de la razón social.

entidades_chip["nombre_chip"] = (
    entidades_chip["razon_social_chip"]
    .str.replace(
        r"\s*\(\s*\d+\s*\)\s*$",
        "",
        regex=True
    )
    .str.strip()
)


# Algunos municipios con nombres repetidos aparecen como:
# La Unión - Antioquia
# La Unión - Valle del Cauca

partes_nombre = (
    entidades_chip["nombre_chip"]
    .str.extract(
        r"^(.*?)\s+-\s+(.+)$"
    )
)

entidades_chip["municipio_chip"] = (
    partes_nombre[0]
    .fillna(entidades_chip["nombre_chip"])
    .str.strip()
)

entidades_chip["departamento_chip"] = (
    partes_nombre[1]
    .fillna("")
    .str.strip()
)

entidades_chip["municipio_norm"] = (
    entidades_chip["municipio_chip"]
    .map(normalizar_texto)
)

entidades_chip["departamento_norm"] = (
    entidades_chip["departamento_chip"]
    .map(normalizar_texto)
)


# ------------------------------------------------------------
# 13. IDENTIFICAR ENTIDADES QUE NO SON MUNICIPIOS
#
# Esto evita asignar, por ejemplo, la Empresa de Desarrollo
# Urbano de Armenia al municipio de Armenia.
# ------------------------------------------------------------

patron_no_municipal = (
    r"\b(?:"
    r"EMPRESA|"
    r"INSTITUTO|"
    r"ASOCIACION|"
    r"CORPORACION|"
    r"DEPARTAMENTO|"
    r"CONTRALORIA|"
    r"PENSIONES?|"
    r"RADIO|"
    r"FONDO|"
    r"HOSPITAL|"
    r"UNIVERSIDAD|"
    r"AGENCIA"
    r")\b"
    r"|\bU A E\b"
)

entidades_chip["es_institucional"] = (
    entidades_chip["municipio_norm"]
    .str.contains(
        patron_no_municipal,
        regex=True,
        na=False
    )
)


# ------------------------------------------------------------
# 14. PRIMER MÉTODO:
#     RECUPERAR EL DANE DESDE EL CÓDIGO CHIP
#
# Para la gran mayoría de municipios, los últimos cinco
# dígitos del código CHIP corresponden al código DANE.
# ------------------------------------------------------------

entidades_chip["codigo_candidato"] = (
    entidades_chip["codigo_entidad_chip"]
    .str.extract(r"(\d{5})$")[0]
)

codigos_validos = set(
    maestro_municipios["codigo_municipio"]
)

entidades_chip["codigo_municipio"] = pd.Series(
    pd.NA,
    index=entidades_chip.index,
    dtype="string"
)

entidades_chip["metodo_enlace"] = (
    "no_identificada"
)

coincide_codigo = (
    entidades_chip["codigo_candidato"]
    .isin(codigos_validos)
    &
    ~entidades_chip["es_institucional"]
)

entidades_chip.loc[
    coincide_codigo,
    "codigo_municipio"
] = entidades_chip.loc[
    coincide_codigo,
    "codigo_candidato"
]

entidades_chip.loc[
    coincide_codigo,
    "metodo_enlace"
] = "codigo_entidad"


# ------------------------------------------------------------
# 15. SEGUNDO MÉTODO:
#     MUNICIPIO + DEPARTAMENTO
# ------------------------------------------------------------

conteo_nombre_departamento = (
    maestro_municipios
    .groupby(
        [
            "municipio_norm",
            "departamento_norm"
        ]
    )
    .size()
)

claves_unicas = (
    conteo_nombre_departamento[
        conteo_nombre_departamento == 1
    ]
    .index
)

maestro_nombre_departamento = (
    maestro_municipios
    .set_index(
        [
            "municipio_norm",
            "departamento_norm"
        ]
    )
    .loc[claves_unicas]
)

diccionario_nombre_departamento = (
    maestro_nombre_departamento[
        "codigo_municipio"
    ]
    .to_dict()
)

pendientes = (
    entidades_chip["codigo_municipio"].isna()
    &
    entidades_chip["departamento_norm"].ne("")
    &
    ~entidades_chip["es_institucional"]
)

for indice in entidades_chip.index[pendientes]:

    clave = (
        entidades_chip.at[
            indice,
            "municipio_norm"
        ],
        entidades_chip.at[
            indice,
            "departamento_norm"
        ]
    )

    codigo_encontrado = (
        diccionario_nombre_departamento
        .get(clave)
    )

    if codigo_encontrado is not None:

        entidades_chip.at[
            indice,
            "codigo_municipio"
        ] = codigo_encontrado

        entidades_chip.at[
            indice,
            "metodo_enlace"
        ] = "nombre_departamento"


# ------------------------------------------------------------
# 16. TERCER MÉTODO:
#     NOMBRE MUNICIPAL ÚNICO EN COLOMBIA
#
# Solo se utiliza cuando el nombre corresponde a un único
# municipio en la DIVIPOLA.
# ------------------------------------------------------------

conteo_nombres = (
    maestro_municipios
    .groupby("municipio_norm")
    ["codigo_municipio"]
    .nunique()
)

nombres_unicos = (
    conteo_nombres[
        conteo_nombres == 1
    ]
    .index
)

maestro_nombres_unicos = (
    maestro_municipios[
        maestro_municipios[
            "municipio_norm"
        ].isin(nombres_unicos)
    ]
    .drop_duplicates(
        subset=["municipio_norm"]
    )
)

diccionario_nombres_unicos = (
    maestro_nombres_unicos
    .set_index("municipio_norm")
    ["codigo_municipio"]
    .to_dict()
)

pendientes = (
    entidades_chip["codigo_municipio"].isna()
    &
    ~entidades_chip["es_institucional"]
)

for indice in entidades_chip.index[pendientes]:

    codigo_encontrado = (
        diccionario_nombres_unicos.get(
            entidades_chip.at[
                indice,
                "municipio_norm"
            ]
        )
    )

    if codigo_encontrado is not None:

        entidades_chip.at[
            indice,
            "codigo_municipio"
        ] = codigo_encontrado

        entidades_chip.at[
            indice,
            "metodo_enlace"
        ] = "nombre_unico"


# ------------------------------------------------------------
# 17. INCORPORAR LOS DATOS DIVIPOLA AL MAPA CHIP
# ------------------------------------------------------------

mapa_chip = entidades_chip.merge(
    maestro_municipios[
        [
            "codigo_departamento",
            "departamento",
            "codigo_municipio",
            "municipio"
        ]
    ],
    on="codigo_municipio",
    how="left",
    validate="many_to_one"
)


# Entidades que no pudieron identificarse como municipios.
# Esta tabla queda disponible para revisión.

entidades_no_identificadas = (
    mapa_chip[
        mapa_chip["codigo_municipio"].isna()
    ][
        [
            "codigo_entidad_chip",
            "nombre_chip",
            "es_institucional",
            "metodo_enlace"
        ]
    ]
    .sort_values("nombre_chip")
    .reset_index(drop=True)
)


# ============================================================
# CONSTRUCCIÓN ANUAL DE L_CHIP Y C_CHIP
# ============================================================


# ------------------------------------------------------------
# 18. CONSERVAR EL CIERRE DEL CUARTO TRIMESTRE
# ------------------------------------------------------------

chip_cierre = chip[
    (chip["fecha"].dt.month == 12)
    &
    (chip["fecha"].dt.day == 31)
].copy()


# ------------------------------------------------------------
# 19. VERIFICAR DUPLICADOS ENTIDAD-AÑO-CUENTA
# ------------------------------------------------------------

duplicados_cierre = chip_cierre[
    chip_cierre.duplicated(
        subset=[
            "codigo_entidad_chip",
            "año",
            "cuenta"
        ],
        keep=False
    )
]

if not duplicados_cierre.empty:

    display(duplicados_cierre)

    raise ValueError(
        "Existen registros duplicados en el cierre anual CHIP."
    )


# ------------------------------------------------------------
# 20. PASAR LAS CUENTAS A COLUMNAS
# ------------------------------------------------------------

chip_anual_componentes = (
    chip_cierre
    .pivot(
        index=[
            "codigo_entidad_chip",
            "año"
        ],
        columns="cuenta",
        values="saldo_final"
    )
    .reset_index()
)

chip_anual_componentes.columns.name = None


# Asegurar que existan las tres columnas.

for cuenta in [
    "1.3.05.07",
    "1.3.10.07",
    "4.1.05.07"
]:

    if cuenta not in chip_anual_componentes.columns:

        chip_anual_componentes[cuenta] = np.nan


chip_anual_componentes = (
    chip_anual_componentes.rename(
        columns={
            "1.3.05.07": "saldo_final_130507",
            "1.3.10.07": "saldo_final_131007",
            "4.1.05.07": "saldo_final_410507"
        }
    )
)


# ------------------------------------------------------------
# 21. CONSTRUIR L_CHIP
#
# L_CHIP es el saldo final de la cuenta 4.1.05.07
# al 31 de diciembre.
# ------------------------------------------------------------

chip_anual_componentes["L_CHIP"] = (
    chip_anual_componentes[
        "saldo_final_410507"
    ]
)


# ------------------------------------------------------------
# 22. CONSTRUIR C_CIERRE_CHIP
#
# C_CIERRE_CHIP representa la cartera existente
# al finalizar cada año.
#
# Hasta 2017:
#   1305.07 = vigencia actual
#   1310.07 = vigencias anteriores
#
# Desde 2018:
#   la cartera se concentra en 1305.07
# ------------------------------------------------------------

chip_anual_componentes[
    "C_CIERRE_CHIP"
] = chip_anual_componentes[
    "saldo_final_130507"
]

periodo_catalogo_antiguo = (
    chip_anual_componentes["año"] <= 2017
)

chip_anual_componentes.loc[
    periodo_catalogo_antiguo,
    "C_CIERRE_CHIP"
] = (
    chip_anual_componentes.loc[
        periodo_catalogo_antiguo,
        [
            "saldo_final_130507",
            "saldo_final_131007"
        ]
    ]
    .sum(
        axis=1,
        min_count=1
    )
)


# ------------------------------------------------------------
# 23. CONSTRUIR C_CHIP
#
# C_CHIP representa la cartera inicial del año t.
#
# La cartera inicial del año t corresponde a la cartera
# existente al cierre del año t-1.
# ------------------------------------------------------------

cartera_inicial = (
    chip_anual_componentes[
        [
            "codigo_entidad_chip",
            "año",
            "C_CIERRE_CHIP"
        ]
    ]
    .copy()
)

cartera_inicial["año"] = (
    cartera_inicial["año"] + 1
)

cartera_inicial = cartera_inicial.rename(
    columns={
        "C_CIERRE_CHIP": "C_CHIP"
    }
)

chip_anual_componentes = (
    chip_anual_componentes.merge(
        cartera_inicial,
        on=[
            "codigo_entidad_chip",
            "año"
        ],
        how="left",
        validate="one_to_one"
    )
)


# ------------------------------------------------------------
# 24. INCORPORAR EL MAPA MUNICIPAL
# ------------------------------------------------------------

chip_anual_componentes = (
    chip_anual_componentes.merge(
        mapa_chip[
            [
                "codigo_entidad_chip",
                "codigo_departamento",
                "departamento",
                "codigo_municipio",
                "municipio",
                "metodo_enlace"
            ]
        ],
        on="codigo_entidad_chip",
        how="left",
        validate="many_to_one"
    )
)


# Conservar únicamente las entidades identificadas
# como municipios.

chip_municipal = (
    chip_anual_componentes[
        chip_anual_componentes[
            "codigo_municipio"
        ].notna()
    ]
    .copy()
)


# ------------------------------------------------------------
# 25. VERIFICAR QUE NO EXISTAN DOS ENTIDADES CHIP PARA
#     EL MISMO MUNICIPIO Y AÑO
# ------------------------------------------------------------

duplicados_municipio_año = (
    chip_municipal[
        chip_municipal.duplicated(
            subset=[
                "codigo_municipio",
                "año"
            ],
            keep=False
        )
    ]
)

if not duplicados_municipio_año.empty:

    display(
        duplicados_municipio_año[
            [
                "codigo_entidad_chip",
                "codigo_municipio",
                "municipio",
                "año",
                "L_CHIP",
                "C_CHIP",
                "C_CIERRE_CHIP"
            ]
        ]
    )

    raise ValueError(
        "Dos entidades CHIP fueron asignadas al mismo "
        "municipio y año. Deben revisarse antes de continuar."
    )


# ------------------------------------------------------------
# 26. PREPARAR LAS VARIABLES PARA EL PANEL
# ------------------------------------------------------------

datos_panel_chip = (
    chip_municipal[
        [
            "codigo_departamento",
            "codigo_municipio",
            "año",
            "L_CHIP",
            "C_CHIP",
            "C_CIERRE_CHIP"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 27. INCORPORAR LAS VARIABLES CHIP AL PANEL
#
# Se eliminan previamente las columnas si el bloque se vuelve
# a ejecutar. Esto evita columnas terminadas en _x y _y.
# ------------------------------------------------------------

numero_observaciones_antes = len(panel)

panel = panel.drop(
    columns=[
        "L_CHIP",
        "C_CHIP",
        "C_CIERRE_CHIP"
    ],
    errors="ignore"
)

panel = panel.merge(
    datos_panel_chip,
    on=[
        "codigo_departamento",
        "codigo_municipio",
        "año"
    ],
    how="left",
    validate="one_to_one"
)

if len(panel) != numero_observaciones_antes:

    raise ValueError(
        "El número de observaciones del panel cambió "
        "después de incorporar CHIP."
    )


# ------------------------------------------------------------
# 28. ORDENAR EL PANEL
# ------------------------------------------------------------

panel = panel.sort_values(
    by=[
        "codigo_departamento",
        "codigo_municipio",
        "año"
    ]
).reset_index(drop=True)


# ============================================================
# VERIFICACIONES
# ============================================================


# ------------------------------------------------------------
# 29. RESUMEN DEL EMPAREJAMIENTO MUNICIPAL
# ------------------------------------------------------------

print("\n========================================")
print("IDENTIFICACIÓN DE ENTIDADES CHIP")
print("========================================")

print(
    "Entidades CHIP totales:",
    len(mapa_chip)
)

print(
    "Entidades identificadas como municipios:",
    mapa_chip["codigo_municipio"].notna().sum()
)

print(
    "Entidades no identificadas:",
    mapa_chip["codigo_municipio"].isna().sum()
)

print("\nMétodos de identificación:")

print(
    mapa_chip["metodo_enlace"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 30. MOSTRAR ENTIDADES NO IDENTIFICADAS
# ------------------------------------------------------------

print("\nEntidades CHIP no incorporadas al panel:")

display(entidades_no_identificadas)


# ------------------------------------------------------------
# 31. RESUMEN ANUAL DE DISPONIBILIDAD
#
# municipios_panel es solamente un conteo del resumen.
# No se agrega como columna al panel.
# ------------------------------------------------------------

resumen_chip = (
    panel
    .groupby(
        "año",
        as_index=False
    )
    .agg(
        municipios_panel=(
            "codigo_municipio",
            "nunique"
        ),
        L_CHIP_disponibles=(
            "L_CHIP",
            "count"
        ),
        C_CHIP_disponibles=(
            "C_CHIP",
            "count"
        ),
        C_CIERRE_CHIP_disponibles=(
            "C_CIERRE_CHIP",
            "count"
        )
    )
)


# ------------------------------------------------------------
# 32. MOSTRAR EL RESULTADO GENERAL
# ------------------------------------------------------------

print("\n========================================")
print("RESULTADO DEL BLOQUE 3")
print("========================================")

print(
    "Observaciones totales del panel:",
    len(panel)
)

print(
    "Observaciones con L_CHIP:",
    panel["L_CHIP"].notna().sum()
)

print(
    "Observaciones con C_CHIP:",
    panel["C_CHIP"].notna().sum()
)

print(
    "Observaciones con C_CIERRE_CHIP:",
    panel["C_CIERRE_CHIP"].notna().sum()
)

print(
    "Valores negativos de L_CHIP:",
    panel["L_CHIP"].lt(0).sum()
)

print(
    "Valores negativos de C_CHIP:",
    panel["C_CHIP"].lt(0).sum()
)

print(
    "Valores negativos de C_CIERRE_CHIP:",
    panel["C_CIERRE_CHIP"].lt(0).sum()
)

display(resumen_chip)


# ------------------------------------------------------------
# 33. COMPROBAR LA RELACIÓN ENTRE LA CARTERA INICIAL
#     Y LA CARTERA DEL CIERRE ANTERIOR
#
# Para cada municipio:
# C_CHIP del año t debe ser igual a C_CIERRE_CHIP
# del año t-1.
# ------------------------------------------------------------

comprobacion_cartera = (
    panel[
        [
            "codigo_municipio",
            "año",
            "C_CHIP",
            "C_CIERRE_CHIP"
        ]
    ]
    .sort_values(
        by=[
            "codigo_municipio",
            "año"
        ]
    )
    .copy()
)

comprobacion_cartera[
    "C_CIERRE_CHIP_AÑO_ANTERIOR"
] = (
    comprobacion_cartera
    .groupby("codigo_municipio")
    ["C_CIERRE_CHIP"]
    .shift(1)
)

casos_comparables_cartera = (
    comprobacion_cartera["C_CHIP"].notna()
    &
    comprobacion_cartera[
        "C_CIERRE_CHIP_AÑO_ANTERIOR"
    ].notna()
)

diferencias_cartera = (
    comprobacion_cartera.loc[
        casos_comparables_cartera,
        "C_CHIP"
    ]
    -
    comprobacion_cartera.loc[
        casos_comparables_cartera,
        "C_CIERRE_CHIP_AÑO_ANTERIOR"
    ]
).abs()

print("\n========================================")
print("COMPROBACIÓN DE CARTERAS")
print("========================================")

print(
    "Observaciones comparables:",
    casos_comparables_cartera.sum()
)

print(
    "Observaciones que coinciden:",
    diferencias_cartera.eq(0).sum()
)

print(
    "Observaciones que no coinciden:",
    diferencias_cartera.ne(0).sum()
)


# ------------------------------------------------------------
# 34. MOSTRAR LOS COMPONENTES CHIP ANUALES
# ------------------------------------------------------------

display(
    chip_anual_componentes.head(30)
)


# ------------------------------------------------------------
# 35. MOSTRAR EL PANEL COMPLETO
# ------------------------------------------------------------

display(panel.head(30))

Archivo CHIP:
/content/drive/MyDrive/Datos/originales/chip/CHIP_CGN_2007_2026T2_EntGob_130507_131007_410507.xls

Cuentas encontradas:
  1.3.05.07
  1.3.10.07
  4.1.05.07

IDENTIFICACIÓN DE ENTIDADES CHIP
Entidades CHIP totales: 1115
Entidades identificadas como municipios: 1103
Entidades no identificadas: 12

Métodos de identificación:
metodo_enlace
codigo_entidad     1097
no_identificada      12
nombre_unico          6
Name: count, dtype: int64

Entidades CHIP no incorporadas al panel:


,codigo_entidad_chip,nombre_chip,es_institucional,metodo_enlace
0,220205999,Asociación de Municipios Antioqueños del Norte...,True,no_identificada
1,923273115,Contraloría Municipal de Palmira,True,no_identificada
2,923273643,Corporación Mixta para el Desarrollo Social y ...,True,no_identificada
3,133176000,Corporación Vallecaucana de las Cuencas Hidrog...,True,no_identificada
4,827770000,Corporación para el Desarrollo Sostenible de L...,True,no_identificada
5,118888000,"Departamento del Archipiélago de San Andrés, P...",True,no_identificada
6,117373000,Departamento del Tolima,True,no_identificada
7,239163001,Empresa de Desarrollo Urbano de Armenia,True,no_identificada
8,123176000,"Instituto del Deporte, la Educación Física y l...",True,no_identificada
9,220805999,Municipios Unidos del Suroriente Antioqueño,False,no_identificada



RESULTADO DEL BLOQUE 3
Observaciones totales del panel: 20957
Observaciones con L_CHIP: 20432
Observaciones con C_CHIP: 19091
Observaciones con C_CIERRE_CHIP: 20245
Valores negativos de L_CHIP: 0
Valores negativos de C_CHIP: 0
Valores negativos de C_CIERRE_CHIP: 0


,año,municipios_panel,L_CHIP_disponibles,C_CHIP_disponibles,C_CIERRE_CHIP_disponibles
0,2007,1103,1049,0,997
1,2008,1103,1059,990,1008
2,2009,1103,1070,1004,1039
3,2010,1103,1065,1029,1056
4,2011,1103,1071,1049,1062
5,2012,1103,1080,1059,1073
6,2013,1103,1074,1071,1068
7,2014,1103,1067,1056,1063
8,2015,1103,1076,1057,1075
9,2016,1103,1079,1071,1083



COMPROBACIÓN DE CARTERAS
Observaciones comparables: 19091
Observaciones que coinciden: 19091
Observaciones que no coinciden: 0


,codigo_entidad_chip,año,saldo_final_130507,saldo_final_131007,saldo_final_410507,L_CHIP,C_CIERRE_CHIP,C_CHIP,codigo_departamento,departamento,codigo_municipio,municipio,metodo_enlace
0,117373000,2016,<NA>,<NA>,0.0,0.0,<NA>,<NA>,NaN,NaN,<NA>,NaN,no_identificada
1,118888000,2007,3063772000.0,16993047000.0,4671158000.0,4671158000.0,20056819000.0,<NA>,NaN,NaN,<NA>,NaN,no_identificada
2,118888000,2008,4067820000.0,11101216000.0,8216146000.0,8216146000.0,15169036000.0,20056819000.0,NaN,NaN,<NA>,NaN,no_identificada
3,118888000,2009,6984459000.0,10290479000.0,7806392000.0,7806392000.0,17274938000.0,15169036000.0,NaN,NaN,<NA>,NaN,no_identificada
4,118888000,2010,11696728000.0,9681765000.0,9544001000.0,9544001000.0,21378493000.0,17274938000.0,NaN,NaN,<NA>,NaN,no_identificada
5,118888000,2011,3799389000.0,25252237000.0,4836086000.0,4836086000.0,29051626000.0,21378493000.0,NaN,NaN,<NA>,NaN,no_identificada
6,118888000,2012,3592637000.0,13338765000.0,5161292000.0,5161292000.0,16931402000.0,29051626000.0,NaN,NaN,<NA>,NaN,no_identificada
7,118888000,2013,3970361000.0,15054004000.0,8297120000.0,8297120000.0,19024365000.0,16931402000.0,NaN,NaN,<NA>,NaN,no_identificada
8,118888000,2014,4047890000.0,43257003000.0,6177537000.0,6177537000.0,47304893000.0,19024365000.0,NaN,NaN,<NA>,NaN,no_identificada
9,118888000,2015,4516854000.0,47288357000.0,11843063000.0,11843063000.0,51805211000.0,47304893000.0,NaN,NaN,<NA>,NaN,no_identificada


,codigo_departamento,departamento,codigo_municipio,municipio,año,R_ACT_FUT,R_ANT_FUT,L_CHIP,C_CHIP,C_CIERRE_CHIP
0,05,ANTIOQUIA,05001,MEDELLÍN,2007,NaN,NaN,274933035000.0,<NA>,159436192000.0
1,05,ANTIOQUIA,05001,MEDELLÍN,2008,NaN,NaN,366671863000.0,159436192000.0,170794237000.0
2,05,ANTIOQUIA,05001,MEDELLÍN,2009,NaN,NaN,366410885000.0,170794237000.0,204772075000.0
3,05,ANTIOQUIA,05001,MEDELLÍN,2010,NaN,NaN,388197844000.0,204772075000.0,229313294000.0
4,05,ANTIOQUIA,05001,MEDELLÍN,2011,NaN,NaN,426247886000.0,229313294000.0,243456864000.0
5,05,ANTIOQUIA,05001,MEDELLÍN,2012,3.760985e+08,4.791869e+07,455349723000.0,243456864000.0,272182803000.0
6,05,ANTIOQUIA,05001,MEDELLÍN,2013,4.576889e+08,8.247165e+07,561463610000.0,272182803000.0,281461808000.0
7,05,ANTIOQUIA,05001,MEDELLÍN,2014,4.935976e+08,6.027679e+07,600316238000.0,281461808000.0,318114186000.0
8,05,ANTIOQUIA,05001,MEDELLÍN,2015,5.226282e+08,7.616222e+07,626565218000.0,318114186000.0,337406053000.0
9,05,ANTIOQUIA,05001,MEDELLÍN,2016,5.517421e+08,6.199670e+07,660356013000.0,337406053000.0,374452020000.0


In [5]:
# ============================================================
# PROYECTO DE TESIS - CLAUDIA YINETH PINEDA CORTINA
#
# BLOQUE 4. RECAUDO TOTAL DEL IPU REPORTADO EN CUIPO
#
# R_TOT_CUIPO = Total Recaudo reportado directamente
#               por CUIPO para el concepto:
#
#               1.1.01.01.200
#               IMPUESTO PREDIAL UNIFICADO
#
# Fuente:
# CUIPO - Ejecución de Ingresos
#
# Periodo disponible: 2021-2025
#
# Este bloque:
# - No suma vigencia actual y vigencias anteriores.
# - No suma recaudo con fondos y sin fondos.
# - No suma urbano y rural.
# - No incluye la sobretasa ambiental.
# - No construye todavía la variable definitiva R_TOT.
# ============================================================


# ------------------------------------------------------------
# 1. DEFINIR LOS AÑOS DISPONIBLES EN CUIPO
# ------------------------------------------------------------

ANIOS_CUIPO = list(
    range(2021, 2026)
)


# ------------------------------------------------------------
# 2. DEFINIR EL CÓDIGO TOTAL DEL IPU EN CUIPO
# ------------------------------------------------------------

CODIGO_IPU_TOTAL_CUIPO = "1.1.01.01.200"


# ------------------------------------------------------------
# 3. CREAR LISTA PARA ALMACENAR LOS RESULTADOS
# ------------------------------------------------------------

bases_recaudo_cuipo = []


# ------------------------------------------------------------
# 4. CÓDIGOS MUNICIPALES VÁLIDOS DEL PANEL
# ------------------------------------------------------------

codigos_municipales_panel = set(
    panel["codigo_municipio"]
    .astype(str)
    .str.strip()
    .str.zfill(5)
    .unique()
)


# ============================================================
# PARTE A. LEER CUIPO 2021-2025
# ============================================================

for año in ANIOS_CUIPO:

    print(f"\nLeyendo CUIPO {año}...")


    # --------------------------------------------------------
    # 5. DEFINIR EL ARCHIVO CORRESPONDIENTE
    # --------------------------------------------------------

    if año == 2025:

        archivo_cuipo = (
            RUTA_PREDIAL
            / "2025_8_CUIPOB_EJECUCION_DE_INGRESOS.xlsb"
        )

        motor_cuipo = "pyxlsb"

    else:

        archivo_cuipo = (
            RUTA_PREDIAL
            / f"{año}_8_CUIPOB_EJECUCION_DE_INGRESOS.xlsx"
        )

        motor_cuipo = None


    # --------------------------------------------------------
    # 6. VERIFICAR QUE EL ARCHIVO EXISTA
    # --------------------------------------------------------

    if not archivo_cuipo.exists():

        raise FileNotFoundError(
            f"No se encontró el archivo CUIPO {año}:\n"
            f"{archivo_cuipo}"
        )

    print("  Archivo:", archivo_cuipo.name)


        # --------------------------------------------------------
    # 7. DEFINIR LOS ARGUMENTOS DE LECTURA
    #
    # CUIPO 2021-2024:
    #   Encabezados en la fila 18 de Excel → header=17
    #
    # CUIPO 2025:
    #   Encabezados en la fila 17 de Excel → header=16
    # --------------------------------------------------------

    if año == 2025:

        fila_encabezado_cuipo = 16

    else:

        fila_encabezado_cuipo = 17


    argumentos_lectura = {
        "io": archivo_cuipo,
        "header": fila_encabezado_cuipo,
        "usecols": [
            "Código DANE",
            "Código Concepto",
            "Concepto",
            "Total Recaudo"
        ],
        "dtype": {
            "Código DANE": str,
            "Código Concepto": str
        }
    }

    if motor_cuipo is not None:

        argumentos_lectura["engine"] = motor_cuipo


    # --------------------------------------------------------
    # 8. LEER EL ARCHIVO
    # --------------------------------------------------------

    cuipo = pd.read_excel(
        **argumentos_lectura
    )


    # --------------------------------------------------------
    # 9. LIMPIAR EL CÓDIGO DEL CONCEPTO
    # --------------------------------------------------------

    cuipo["Código Concepto"] = (
        cuipo["Código Concepto"]
        .astype(str)
        .str.strip()
    )


    # --------------------------------------------------------
    # 10. CONSERVAR ÚNICAMENTE LA FILA TOTAL DEL IPU
    #
    # No se seleccionan los códigos hijos:
    # 1.1.01.01.200.01 = IPU urbano
    # 1.1.01.01.200.02 = IPU rural
    #
    # Tampoco se selecciona:
    # 1.1.01.01.014 = Sobretasa ambiental
    # --------------------------------------------------------

    cuipo = cuipo[
        cuipo["Código Concepto"].eq(
            CODIGO_IPU_TOTAL_CUIPO
        )
    ].copy()


    # --------------------------------------------------------
    # 11. CONSTRUIR EL CÓDIGO MUNICIPAL
    # --------------------------------------------------------

    cuipo["codigo_municipio"] = (
        cuipo["Código DANE"]
        .astype(str)
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
        .str.zfill(5)
    )


    # Conservar solamente los municipios del panel.
    # Esto excluye totales departamentales y otras entidades.

    cuipo = cuipo[
        cuipo["codigo_municipio"].isin(
            codigos_municipales_panel
        )
    ].copy()


    # --------------------------------------------------------
    # 12. EXTRAER DIRECTAMENTE TOTAL RECAUDO
    #
    # No se realiza ninguna suma.
    # Únicamente se cambia el nombre de la variable.
    # --------------------------------------------------------

    cuipo["R_TOT_CUIPO"] = a_numerico(
        cuipo["Total Recaudo"]
    )

    cuipo["año"] = año


    # --------------------------------------------------------
    # 13. VERIFICAR DUPLICADOS MUNICIPIO-AÑO
    #
    # Si aparece más de una fila total para el mismo
    # municipio, el código se detiene. No se suman.
    # --------------------------------------------------------

    duplicados_cuipo = cuipo[
        cuipo.duplicated(
            subset=[
                "codigo_municipio",
                "año"
            ],
            keep=False
        )
    ]

    if not duplicados_cuipo.empty:

        print(
            f"\nCUIPO {año}: filas totales duplicadas:"
        )

        display(
            duplicados_cuipo[
                [
                    "codigo_municipio",
                    "año",
                    "Código Concepto",
                    "Concepto",
                    "Total Recaudo"
                ]
            ]
        )

        raise ValueError(
            f"CUIPO {año}: se encontró más de una "
            "fila total del IPU para un mismo municipio."
        )


    # --------------------------------------------------------
    # 14. CONSERVAR SOLAMENTE LAS VARIABLES NECESARIAS
    # --------------------------------------------------------

    cuipo = cuipo[
        [
            "codigo_municipio",
            "año",
            "R_TOT_CUIPO"
        ]
    ].copy()


    print(
        "  Municipios con R_TOT_CUIPO:",
        cuipo["R_TOT_CUIPO"].notna().sum()
    )

    print(
        "  Valores iguales a cero:",
        cuipo["R_TOT_CUIPO"].eq(0).sum()
    )


    bases_recaudo_cuipo.append(
        cuipo
    )


# ============================================================
# PARTE B. CONSOLIDAR CUIPO 2021-2025
# ============================================================


# ------------------------------------------------------------
# 15. UNIR LOS CINCO AÑOS
# ------------------------------------------------------------

recaudo_cuipo = pd.concat(
    bases_recaudo_cuipo,
    ignore_index=True
)


# ------------------------------------------------------------
# 16. VERIFICAR UNICIDAD EN LA BASE COMPLETA
# ------------------------------------------------------------

duplicados_cuipo_total = (
    recaudo_cuipo.duplicated(
        subset=[
            "codigo_municipio",
            "año"
        ]
    ).sum()
)

if duplicados_cuipo_total > 0:

    raise ValueError(
        "La base CUIPO consolidada contiene "
        f"{duplicados_cuipo_total} duplicados municipio-año."
    )


# ============================================================
# PARTE C. INCORPORAR R_TOT_CUIPO AL PANEL
# ============================================================


# ------------------------------------------------------------
# 17. ELIMINAR COLUMNAS DE PRUEBAS ANTERIORES
#
# Esto permite volver a ejecutar el bloque sin generar
# columnas terminadas en _x y _y.
# ------------------------------------------------------------

numero_observaciones_antes = len(panel)

panel = panel.drop(
    columns=[
        "R_TOT_CUIPO",
        "R_TOT_OEC",
        "R_TOT"
    ],
    errors="ignore"
)


# ------------------------------------------------------------
# 18. INCORPORAR R_TOT_CUIPO
# ------------------------------------------------------------

panel = panel.merge(
    recaudo_cuipo,
    on=[
        "codigo_municipio",
        "año"
    ],
    how="left",
    validate="one_to_one"
)


# ------------------------------------------------------------
# 19. VERIFICAR QUE EL PANEL CONSERVE SU TAMAÑO
# ------------------------------------------------------------

if len(panel) != numero_observaciones_antes:

    raise ValueError(
        "El número de observaciones del panel cambió "
        "después de incorporar R_TOT_CUIPO."
    )


# ------------------------------------------------------------
# 20. ORDENAR EL PANEL
# ------------------------------------------------------------

panel = panel.sort_values(
    by=[
        "codigo_departamento",
        "codigo_municipio",
        "año"
    ]
).reset_index(drop=True)


# ============================================================
# VERIFICACIONES
# ============================================================


# ------------------------------------------------------------
# 21. RESUMEN GENERAL
# ------------------------------------------------------------

print("\n========================================")
print("RESULTADO DEL BLOQUE 4")
print("========================================")

print(
    "Observaciones totales del panel:",
    len(panel)
)

print(
    "R_TOT_CUIPO disponibles:",
    panel["R_TOT_CUIPO"].notna().sum()
)

print(
    "R_TOT_CUIPO iguales a cero:",
    panel["R_TOT_CUIPO"].eq(0).sum()
)

print(
    "R_TOT_CUIPO negativos:",
    panel["R_TOT_CUIPO"].lt(0).sum()
)


# ------------------------------------------------------------
# 22. DISPONIBILIDAD POR AÑO
#
# El resumen se limita a 2021-2025.
# municipios_panel no se agrega como columna.
# ------------------------------------------------------------

resumen_cuipo = (
    panel[
        panel["año"].between(2021, 2025)
    ]
    .groupby(
        "año",
        as_index=False
    )
    .agg(
        municipios_panel=(
            "codigo_municipio",
            "nunique"
        ),
        R_TOT_CUIPO_disponibles=(
            "R_TOT_CUIPO",
            "count"
        ),
        R_TOT_CUIPO_ceros=(
            "R_TOT_CUIPO",
            lambda serie: serie.eq(0).sum()
        )
    )
)

display(resumen_cuipo)


# ------------------------------------------------------------
# 23. OBSERVACIONES SIN INFORMACIÓN CUIPO
# ------------------------------------------------------------

observaciones_sin_cuipo = (
    panel[
        panel["año"].between(2021, 2025)
        &
        panel["R_TOT_CUIPO"].isna()
    ][
        [
            "codigo_departamento",
            "departamento",
            "codigo_municipio",
            "municipio",
            "año"
        ]
    ]
    .reset_index(drop=True)
)

print("\nObservaciones 2021-2025 sin R_TOT_CUIPO:")

display(observaciones_sin_cuipo)


# ------------------------------------------------------------
# 24. MOSTRAR CASOS DE COMPROBACIÓN
# ------------------------------------------------------------

comprobacion_cuipo = (
    panel[
        panel["codigo_municipio"].isin(
            [
                "05001",  # Medellín
                "05002"   # Abejorral
            ]
        )
        &
        panel["año"].between(2021, 2025)
    ][
        [
            "codigo_municipio",
            "municipio",
            "año",
            "R_TOT_CUIPO"
        ]
    ]
)

display(comprobacion_cuipo)


# ------------------------------------------------------------
# 25. MOSTRAR EL PANEL
# ------------------------------------------------------------

display(panel.head(30))


Leyendo CUIPO 2021...
  Archivo: 2021_8_CUIPOB_EJECUCION_DE_INGRESOS.xlsx
  Municipios con R_TOT_CUIPO: 1084
  Valores iguales a cero: 1

Leyendo CUIPO 2022...
  Archivo: 2022_8_CUIPOB_EJECUCION_DE_INGRESOS.xlsx
  Municipios con R_TOT_CUIPO: 1086
  Valores iguales a cero: 2

Leyendo CUIPO 2023...
  Archivo: 2023_8_CUIPOB_EJECUCION_DE_INGRESOS.xlsx
  Municipios con R_TOT_CUIPO: 1062
  Valores iguales a cero: 0

Leyendo CUIPO 2024...
  Archivo: 2024_8_CUIPOB_EJECUCION_DE_INGRESOS.xlsx
  Municipios con R_TOT_CUIPO: 1090
  Valores iguales a cero: 0

Leyendo CUIPO 2025...
  Archivo: 2025_8_CUIPOB_EJECUCION_DE_INGRESOS.xlsb
  Municipios con R_TOT_CUIPO: 1087
  Valores iguales a cero: 0

RESULTADO DEL BLOQUE 4
Observaciones totales del panel: 20957
R_TOT_CUIPO disponibles: 5409
R_TOT_CUIPO iguales a cero: 3
R_TOT_CUIPO negativos: 0


,año,municipios_panel,R_TOT_CUIPO_disponibles,R_TOT_CUIPO_ceros
0,2021,1103,1084,1
1,2022,1103,1086,2
2,2023,1103,1062,0
3,2024,1103,1090,0
4,2025,1103,1087,0



Observaciones 2021-2025 sin R_TOT_CUIPO:


,codigo_departamento,departamento,codigo_municipio,municipio,año
0,05,ANTIOQUIA,05150,CAROLINA,2023
1,05,ANTIOQUIA,05664,SAN PEDRO DE LOS MILAGROS,2023
2,08,ATLÁNTICO,08558,POLONUEVO,2023
3,13,BOLÍVAR,13030,ALTOS DEL ROSARIO,2023
4,13,BOLÍVAR,13442,MARÍA LA BAJA,2022
...,...,...,...,...,...
101,97,VAUPÉS,97666,TARAIRA,2021
102,97,VAUPÉS,97666,TARAIRA,2022
103,97,VAUPÉS,97666,TARAIRA,2023
104,97,VAUPÉS,97666,TARAIRA,2024


,codigo_municipio,municipio,año,R_TOT_CUIPO
14,05001,MEDELLÍN,2021,8.352043e+11
15,05001,MEDELLÍN,2022,8.697222e+11
16,05001,MEDELLÍN,2023,9.831456e+11
17,05001,MEDELLÍN,2024,1.163170e+12
18,05001,MEDELLÍN,2025,1.272699e+12
33,05002,ABEJORRAL,2021,1.066949e+09
34,05002,ABEJORRAL,2022,9.935449e+08
35,05002,ABEJORRAL,2023,1.006257e+09
36,05002,ABEJORRAL,2024,1.138352e+09
37,05002,ABEJORRAL,2025,1.575455e+09


,codigo_departamento,departamento,codigo_municipio,municipio,año,R_ACT_FUT,R_ANT_FUT,L_CHIP,C_CHIP,C_CIERRE_CHIP,R_TOT_CUIPO
0,05,ANTIOQUIA,05001,MEDELLÍN,2007,NaN,NaN,274933035000.0,<NA>,159436192000.0,NaN
1,05,ANTIOQUIA,05001,MEDELLÍN,2008,NaN,NaN,366671863000.0,159436192000.0,170794237000.0,NaN
2,05,ANTIOQUIA,05001,MEDELLÍN,2009,NaN,NaN,366410885000.0,170794237000.0,204772075000.0,NaN
3,05,ANTIOQUIA,05001,MEDELLÍN,2010,NaN,NaN,388197844000.0,204772075000.0,229313294000.0,NaN
4,05,ANTIOQUIA,05001,MEDELLÍN,2011,NaN,NaN,426247886000.0,229313294000.0,243456864000.0,NaN
5,05,ANTIOQUIA,05001,MEDELLÍN,2012,3.760985e+08,4.791869e+07,455349723000.0,243456864000.0,272182803000.0,NaN
6,05,ANTIOQUIA,05001,MEDELLÍN,2013,4.576889e+08,8.247165e+07,561463610000.0,272182803000.0,281461808000.0,NaN
7,05,ANTIOQUIA,05001,MEDELLÍN,2014,4.935976e+08,6.027679e+07,600316238000.0,281461808000.0,318114186000.0,NaN
8,05,ANTIOQUIA,05001,MEDELLÍN,2015,5.226282e+08,7.616222e+07,626565218000.0,318114186000.0,337406053000.0,NaN
9,05,ANTIOQUIA,05001,MEDELLÍN,2016,5.517421e+08,6.199670e+07,660356013000.0,337406053000.0,374452020000.0,NaN


In [7]:
# ============================================================
# PROYECTO DE TESIS - CLAUDIA YINETH PINEDA CORTINA
#
# BLOQUE 5. CONSTRUCCIÓN DEL RECAUDO TOTAL DEL IPU
#
# R_TOT_FUT:
#   Recaudo de vigencia actual + vigencias anteriores
#   reportado en FUT para 2012-2020.
#
# R_TOT:
#   2012-2020 = R_TOT_FUT
#   2021-2025 = R_TOT_CUIPO
#
# No se realizan imputaciones.
# No se utiliza la información OEC.
# ============================================================


# ------------------------------------------------------------
# 1. VERIFICAR LAS VARIABLES NECESARIAS
# ------------------------------------------------------------

columnas_requeridas_r_tot = [
    "R_ACT_FUT",
    "R_ANT_FUT",
    "R_TOT_CUIPO"
]

columnas_faltantes_r_tot = [
    columna
    for columna in columnas_requeridas_r_tot
    if columna not in panel.columns
]

if columnas_faltantes_r_tot:

    raise ValueError(
        "Faltan las siguientes variables para construir "
        f"R_TOT:\n{columnas_faltantes_r_tot}"
    )


# ------------------------------------------------------------
# 2. ELIMINAR RESULTADOS DE EJECUCIONES ANTERIORES
#
# Esto permite volver a ejecutar el bloque sin generar
# columnas duplicadas.
# ------------------------------------------------------------

panel = panel.drop(
    columns=[
        "R_TOT_FUT",
        "R_TOT"
    ],
    errors="ignore"
)


# ------------------------------------------------------------
# 3. ASEGURAR FORMATO NUMÉRICO
# ------------------------------------------------------------

for columna in columnas_requeridas_r_tot:

    panel[columna] = pd.to_numeric(
        panel[columna],
        errors="coerce"
    )


# ------------------------------------------------------------
# 4. DEFINIR EL FACTOR DE ESCALA DEL FUT
#
# FUT 2012-2016:
#   Los valores se encuentran expresados en miles de pesos.
#
# FUT 2017-2020:
#   Los valores se encuentran expresados en pesos.
#
# El factor se utiliza solamente durante el cálculo.
# No se agrega como columna al panel.
# ------------------------------------------------------------

factor_escala_fut = pd.Series(
    np.where(
        panel["año"].between(2012, 2016),
        1000.0,
        1.0
    ),
    index=panel.index
)


# ------------------------------------------------------------
# 5. IDENTIFICAR OBSERVACIONES FUT COMPLETAS
#
# Para construir R_TOT_FUT deben estar disponibles tanto
# R_ACT_FUT como R_ANT_FUT.
#
# Un valor ausente no se reemplaza por cero.
# ------------------------------------------------------------

periodo_fut = (
    panel["año"].between(2012, 2020)
)

casos_fut_completos = (
    periodo_fut
    &
    panel["R_ACT_FUT"].notna()
    &
    panel["R_ANT_FUT"].notna()
)


# ------------------------------------------------------------
# 6. CONSTRUIR R_TOT_FUT
# ------------------------------------------------------------

panel["R_TOT_FUT"] = np.nan

panel.loc[
    casos_fut_completos,
    "R_TOT_FUT"
] = (
    (
        panel.loc[
            casos_fut_completos,
            "R_ACT_FUT"
        ]
        +
        panel.loc[
            casos_fut_completos,
            "R_ANT_FUT"
        ]
    )
    *
    factor_escala_fut.loc[
        casos_fut_completos
    ]
)


# ------------------------------------------------------------
# 7. CONSTRUIR R_TOT
#
# 2012-2020:
#   Se utiliza R_TOT_FUT.
#
# 2021-2025:
#   Se utiliza R_TOT_CUIPO.
# ------------------------------------------------------------

panel["R_TOT"] = np.nan

periodo_r_tot_fut = (
    panel["año"].between(2012, 2020)
)

periodo_r_tot_cuipo = (
    panel["año"].between(2021, 2025)
)

panel.loc[
    periodo_r_tot_fut,
    "R_TOT"
] = panel.loc[
    periodo_r_tot_fut,
    "R_TOT_FUT"
]

panel.loc[
    periodo_r_tot_cuipo,
    "R_TOT"
] = panel.loc[
    periodo_r_tot_cuipo,
    "R_TOT_CUIPO"
]


# ------------------------------------------------------------
# 8. VERIFICAR VALORES NO VÁLIDOS
# ------------------------------------------------------------

if np.isinf(
    panel[
        [
            "R_TOT_FUT",
            "R_TOT"
        ]
    ].to_numpy()
).any():

    raise ValueError(
        "Se encontraron valores infinitos en las variables "
        "de recaudo total."
    )


# ------------------------------------------------------------
# 9. RESUMEN DE DISPONIBILIDAD POR AÑO
# ------------------------------------------------------------

resumen_r_tot = (
    panel[
        panel["año"].between(2012, 2025)
    ]
    .groupby(
        "año",
        as_index=False
    )
    .agg(
        municipios_panel=(
            "codigo_municipio",
            "nunique"
        ),
        R_ACT_FUT_disponibles=(
            "R_ACT_FUT",
            "count"
        ),
        R_ANT_FUT_disponibles=(
            "R_ANT_FUT",
            "count"
        ),
        R_TOT_FUT_disponibles=(
            "R_TOT_FUT",
            "count"
        ),
        R_TOT_CUIPO_disponibles=(
            "R_TOT_CUIPO",
            "count"
        ),
        R_TOT_disponibles=(
            "R_TOT",
            "count"
        )
    )
)


# ------------------------------------------------------------
# 10. MOSTRAR RESULTADOS GENERALES
# ------------------------------------------------------------

print("\n========================================")
print("RESULTADO DEL BLOQUE 5")
print("========================================")

print(
    "Observaciones con R_TOT_FUT:",
    panel["R_TOT_FUT"].notna().sum()
)

print(
    "Observaciones con R_TOT_CUIPO:",
    panel["R_TOT_CUIPO"].notna().sum()
)

print(
    "Observaciones con R_TOT:",
    panel["R_TOT"].notna().sum()
)

print(
    "R_TOT iguales a cero:",
    panel["R_TOT"].eq(0).sum()
)

print(
    "R_TOT negativos:",
    panel["R_TOT"].lt(0).sum()
)

print(
    "Observaciones FUT 2012-2020 sin los dos componentes:",
    (
        periodo_fut
        &
        ~casos_fut_completos
    ).sum()
)

display(resumen_r_tot)


# ------------------------------------------------------------
# 11. MOSTRAR CASOS DE COMPROBACIÓN
# ------------------------------------------------------------

comprobacion_r_tot = (
    panel[
        panel["codigo_municipio"].isin(
            [
                "05001",  # Medellín
                "05002"   # Abejorral
            ]
        )
        &
        panel["año"].between(2012, 2025)
    ][
        [
            "codigo_municipio",
            "municipio",
            "año",
            "R_ACT_FUT",
            "R_ANT_FUT",
            "R_TOT_FUT",
            "R_TOT_CUIPO",
            "R_TOT"
        ]
    ]
)

display(comprobacion_r_tot)


# ------------------------------------------------------------
# 12. MOSTRAR EL PANEL
# ------------------------------------------------------------

display(panel.head(30))


RESULTADO DEL BLOQUE 5
Observaciones con R_TOT_FUT: 8083
Observaciones con R_TOT_CUIPO: 5409
Observaciones con R_TOT: 13492
R_TOT iguales a cero: 49
R_TOT negativos: 0
Observaciones FUT 2012-2020 sin los dos componentes: 1844


,año,municipios_panel,R_ACT_FUT_disponibles,R_ANT_FUT_disponibles,R_TOT_FUT_disponibles,R_TOT_CUIPO_disponibles,R_TOT_disponibles
0,2012,1103,1069,832,829,0,829
1,2013,1103,1082,886,883,0,883
2,2014,1103,1082,908,904,0,904
3,2015,1103,1025,842,831,0,831
4,2016,1103,1069,879,872,0,872
5,2017,1103,1076,933,927,0,927
6,2018,1103,1084,953,950,0,950
7,2019,1103,1081,952,946,0,946
8,2020,1103,1087,949,941,0,941
9,2021,1103,0,0,0,1084,1084


,codigo_municipio,municipio,año,R_ACT_FUT,R_ANT_FUT,R_TOT_FUT,R_TOT_CUIPO,R_TOT
5,05001,MEDELLÍN,2012,3.760985e+08,4.791869e+07,4.240172e+11,NaN,4.240172e+11
6,05001,MEDELLÍN,2013,4.576889e+08,8.247165e+07,5.401606e+11,NaN,5.401606e+11
7,05001,MEDELLÍN,2014,4.935976e+08,6.027679e+07,5.538744e+11,NaN,5.538744e+11
8,05001,MEDELLÍN,2015,5.226282e+08,7.616222e+07,5.987905e+11,NaN,5.987905e+11
9,05001,MEDELLÍN,2016,5.517421e+08,6.199670e+07,6.137388e+11,NaN,6.137388e+11
10,05001,MEDELLÍN,2017,5.897003e+11,6.933346e+10,6.590338e+11,NaN,6.590338e+11
11,05001,MEDELLÍN,2018,6.166994e+11,6.674290e+10,6.834423e+11,NaN,6.834423e+11
12,05001,MEDELLÍN,2019,6.637510e+11,8.875013e+10,7.525012e+11,NaN,7.525012e+11
13,05001,MEDELLÍN,2020,6.708886e+11,1.045724e+11,7.754609e+11,NaN,7.754609e+11
14,05001,MEDELLÍN,2021,NaN,NaN,NaN,8.352043e+11,8.352043e+11


,codigo_departamento,departamento,codigo_municipio,municipio,año,R_ACT_FUT,R_ANT_FUT,L_CHIP,C_CHIP,C_CIERRE_CHIP,R_TOT_CUIPO,R_TOT_FUT,R_TOT
0,05,ANTIOQUIA,05001,MEDELLÍN,2007,NaN,NaN,274933035000.0,<NA>,159436192000.0,NaN,NaN,NaN
1,05,ANTIOQUIA,05001,MEDELLÍN,2008,NaN,NaN,366671863000.0,159436192000.0,170794237000.0,NaN,NaN,NaN
2,05,ANTIOQUIA,05001,MEDELLÍN,2009,NaN,NaN,366410885000.0,170794237000.0,204772075000.0,NaN,NaN,NaN
3,05,ANTIOQUIA,05001,MEDELLÍN,2010,NaN,NaN,388197844000.0,204772075000.0,229313294000.0,NaN,NaN,NaN
4,05,ANTIOQUIA,05001,MEDELLÍN,2011,NaN,NaN,426247886000.0,229313294000.0,243456864000.0,NaN,NaN,NaN
5,05,ANTIOQUIA,05001,MEDELLÍN,2012,3.760985e+08,4.791869e+07,455349723000.0,243456864000.0,272182803000.0,NaN,4.240172e+11,4.240172e+11
6,05,ANTIOQUIA,05001,MEDELLÍN,2013,4.576889e+08,8.247165e+07,561463610000.0,272182803000.0,281461808000.0,NaN,5.401606e+11,5.401606e+11
7,05,ANTIOQUIA,05001,MEDELLÍN,2014,4.935976e+08,6.027679e+07,600316238000.0,281461808000.0,318114186000.0,NaN,5.538744e+11,5.538744e+11
8,05,ANTIOQUIA,05001,MEDELLÍN,2015,5.226282e+08,7.616222e+07,626565218000.0,318114186000.0,337406053000.0,NaN,5.987905e+11,5.987905e+11
9,05,ANTIOQUIA,05001,MEDELLÍN,2016,5.517421e+08,6.199670e+07,660356013000.0,337406053000.0,374452020000.0,NaN,6.137388e+11,6.137388e+11


In [8]:
# ============================================================
# PROYECTO DE TESIS - CLAUDIA YINETH PINEDA CORTINA
#
# BLOQUE 6. CONSTRUCCIÓN DE LAS VARIABLES DE RESULTADO
#
# y_C:
# Cumplimiento predial global observado.
#
#       R_TOT
# -------------------
#   C_CHIP + L_CHIP
#
#
# y_R:
# Reducción neta observada de la cartera.
#
#   C_CHIP - C_CIERRE_CHIP
# ---------------------------
#           C_CHIP
#
#
# La letra y minúscula representa el resultado observado.
# La letra Y mayúscula se reservará para los resultados
# potenciales en la formulación del modelo causal.
#
# En este bloque:
# - No se imputan valores ausentes.
# - No se reemplazan valores por cero.
# - No se eliminan valores extremos.
# - No se limita y_C al intervalo entre 0 y 1.
# ============================================================


# ------------------------------------------------------------
# 1. VERIFICAR LAS VARIABLES NECESARIAS
# ------------------------------------------------------------

columnas_requeridas_resultados = [
    "R_TOT",
    "L_CHIP",
    "C_CHIP",
    "C_CIERRE_CHIP"
]

columnas_faltantes_resultados = [
    columna
    for columna in columnas_requeridas_resultados
    if columna not in panel.columns
]

if columnas_faltantes_resultados:

    raise ValueError(
        "Faltan las siguientes variables para construir "
        f"y_C y y_R:\n{columnas_faltantes_resultados}"
    )


# ------------------------------------------------------------
# 2. ELIMINAR RESULTADOS DE EJECUCIONES ANTERIORES
#
# También se eliminan Y_C y Y_R, en caso de que se haya
# ejecutado anteriormente la versión con mayúsculas.
# ------------------------------------------------------------

panel = panel.drop(
    columns=[
        "Y_C",
        "Y_R",
        "y_C",
        "y_R"
    ],
    errors="ignore"
)


# ------------------------------------------------------------
# 3. ASEGURAR FORMATO NUMÉRICO
# ------------------------------------------------------------

for columna in columnas_requeridas_resultados:

    panel[columna] = pd.to_numeric(
        panel[columna],
        errors="coerce"
    )


# ============================================================
# PARTE A. CUMPLIMIENTO PREDIAL GLOBAL
# ============================================================


# ------------------------------------------------------------
# 4. CALCULAR EL TOTAL DISPONIBLE PARA COBRO
#
# Se utiliza únicamente durante el cálculo.
# No se agrega como columna al panel.
# ------------------------------------------------------------

total_disponible_cobro = (
    panel["C_CHIP"]
    +
    panel["L_CHIP"]
)


# ------------------------------------------------------------
# 5. IDENTIFICAR OBSERVACIONES VÁLIDAS PARA y_C
#
# Los tres componentes deben estar disponibles y el
# denominador debe ser mayor que cero.
# ------------------------------------------------------------

casos_validos_y_C = (
    panel["R_TOT"].notna()
    &
    panel["C_CHIP"].notna()
    &
    panel["L_CHIP"].notna()
    &
    total_disponible_cobro.gt(0)
)


# ------------------------------------------------------------
# 6. CONSTRUIR y_C
#
# y_C indica qué proporción de la cartera inicial y del
# predial causado durante el año fue efectivamente recaudada.
# ------------------------------------------------------------

panel["y_C"] = np.nan

panel.loc[
    casos_validos_y_C,
    "y_C"
] = (
    panel.loc[
        casos_validos_y_C,
        "R_TOT"
    ]
    /
    total_disponible_cobro.loc[
        casos_validos_y_C
    ]
)


# ============================================================
# PARTE B. REDUCCIÓN NETA DE LA CARTERA
# ============================================================


# ------------------------------------------------------------
# 7. IDENTIFICAR OBSERVACIONES VÁLIDAS PARA y_R
#
# La cartera inicial y la cartera al cierre deben estar
# disponibles. La cartera inicial debe ser mayor que cero.
# ------------------------------------------------------------

casos_validos_y_R = (
    panel["C_CHIP"].notna()
    &
    panel["C_CIERRE_CHIP"].notna()
    &
    panel["C_CHIP"].gt(0)
)


# ------------------------------------------------------------
# 8. CONSTRUIR y_R
#
# Un valor positivo indica reducción neta de la cartera.
# Un valor igual a cero indica que la cartera no cambió.
# Un valor negativo indica crecimiento de la cartera.
# ------------------------------------------------------------

panel["y_R"] = np.nan

panel.loc[
    casos_validos_y_R,
    "y_R"
] = (
    (
        panel.loc[
            casos_validos_y_R,
            "C_CHIP"
        ]
        -
        panel.loc[
            casos_validos_y_R,
            "C_CIERRE_CHIP"
        ]
    )
    /
    panel.loc[
        casos_validos_y_R,
        "C_CHIP"
    ]
)


# ------------------------------------------------------------
# 9. VERIFICAR VALORES INFINITOS
# ------------------------------------------------------------

if np.isinf(
    panel[
        [
            "y_C",
            "y_R"
        ]
    ].to_numpy()
).any():

    raise ValueError(
        "Se encontraron valores infinitos en y_C o y_R."
    )


# ------------------------------------------------------------
# 10. IDENTIFICAR DENOMINADORES IGUALES A CERO
#
# Estos casos permanecen ausentes en las variables de
# resultado. No se reemplazan ni se eliminan.
# ------------------------------------------------------------

denominadores_y_C_cero = (
    panel["R_TOT"].notna()
    &
    panel["C_CHIP"].notna()
    &
    panel["L_CHIP"].notna()
    &
    total_disponible_cobro.eq(0)
)

denominadores_y_R_cero = (
    panel["C_CHIP"].eq(0)
    &
    panel["C_CIERRE_CHIP"].notna()
)


# ------------------------------------------------------------
# 11. RESUMEN POR AÑO
# ------------------------------------------------------------

resumen_variables_resultado = (
    panel[
        panel["año"].between(2012, 2025)
    ]
    .groupby(
        "año",
        as_index=False
    )
    .agg(
        municipios_panel=(
            "codigo_municipio",
            "nunique"
        ),
        y_C_disponibles=(
            "y_C",
            "count"
        ),
        y_C_mayores_que_uno=(
            "y_C",
            lambda serie: serie.gt(1).sum()
        ),
        y_C_negativos=(
            "y_C",
            lambda serie: serie.lt(0).sum()
        ),
        y_R_disponibles=(
            "y_R",
            "count"
        ),
        y_R_positivos=(
            "y_R",
            lambda serie: serie.gt(0).sum()
        ),
        y_R_negativos=(
            "y_R",
            lambda serie: serie.lt(0).sum()
        )
    )
)


# ------------------------------------------------------------
# 12. MOSTRAR RESULTADOS GENERALES
# ------------------------------------------------------------

print("\n========================================")
print("RESULTADO DEL BLOQUE 6")
print("========================================")

print(
    "Observaciones con y_C:",
    panel["y_C"].notna().sum()
)

print(
    "Observaciones con y_R:",
    panel["y_R"].notna().sum()
)

print(
    "y_C mayores que uno:",
    panel["y_C"].gt(1).sum()
)

print(
    "y_C negativos:",
    panel["y_C"].lt(0).sum()
)

print(
    "y_R positivos:",
    panel["y_R"].gt(0).sum()
)

print(
    "y_R iguales a cero:",
    panel["y_R"].eq(0).sum()
)

print(
    "y_R negativos:",
    panel["y_R"].lt(0).sum()
)

print(
    "Denominadores de y_C iguales a cero:",
    denominadores_y_C_cero.sum()
)

print(
    "Denominadores de y_R iguales a cero:",
    denominadores_y_R_cero.sum()
)

display(resumen_variables_resultado)


# ------------------------------------------------------------
# 13. MOSTRAR CASOS DE COMPROBACIÓN
# ------------------------------------------------------------

comprobacion_variables_resultado = (
    panel[
        panel["codigo_municipio"].isin(
            [
                "05001",  # Medellín
                "05002"   # Abejorral
            ]
        )
        &
        panel["año"].between(2012, 2025)
    ][
        [
            "codigo_municipio",
            "municipio",
            "año",
            "R_TOT",
            "L_CHIP",
            "C_CHIP",
            "C_CIERRE_CHIP",
            "y_C",
            "y_R"
        ]
    ]
)

display(comprobacion_variables_resultado)


# ------------------------------------------------------------
# 14. MOSTRAR EL PANEL COMPLETO
# ------------------------------------------------------------

display(panel.head(30))


RESULTADO DEL BLOQUE 6
Observaciones con y_C: 13180
Observaciones con y_R: 17804
y_C mayores que uno: 266
y_C negativos: 0
y_R positivos: 7032
y_R iguales a cero: 2034
y_R negativos: 8738
Denominadores de y_C iguales a cero: 2
Denominadores de y_R iguales a cero: 1183


,año,municipios_panel,y_C_disponibles,y_C_mayores_que_uno,y_C_negativos,y_R_disponibles,y_R_positivos,y_R_negativos
0,2012,1103,806,20,0,997,394,497
1,2013,1103,860,21,0,1018,435,460
2,2014,1103,878,12,0,1005,366,530
3,2015,1103,813,15,0,991,388,502
4,2016,1103,856,21,0,1000,394,516
5,2017,1103,913,17,0,1016,442,536
6,2018,1103,925,13,0,1008,480,467
7,2019,1103,925,25,0,984,397,480
8,2020,1103,922,20,0,984,488,388
9,2021,1103,1052,26,0,968,407,452


,codigo_municipio,municipio,año,R_TOT,L_CHIP,C_CHIP,C_CIERRE_CHIP,y_C,y_R
5,05001,MEDELLÍN,2012,4.240172e+11,455349723000.0,243456864000.0,272182803000.0,0.606773,-0.117992
6,05001,MEDELLÍN,2013,5.401606e+11,561463610000.0,272182803000.0,281461808000.0,0.647949,-0.034091
7,05001,MEDELLÍN,2014,5.538744e+11,600316238000.0,281461808000.0,318114186000.0,0.628134,-0.130221
8,05001,MEDELLÍN,2015,5.987905e+11,626565218000.0,318114186000.0,337406053000.0,0.633856,-0.060644
9,05001,MEDELLÍN,2016,6.137388e+11,660356013000.0,337406053000.0,374452020000.0,0.615115,-0.109796
10,05001,MEDELLÍN,2017,6.590338e+11,700540439429.0,374452020000.0,409534930460.0,0.613059,-0.093691
11,05001,MEDELLÍN,2018,6.834423e+11,731861284438.0,409534930460.0,189182859744.0,0.598777,0.538054
12,05001,MEDELLÍN,2019,7.525012e+11,788452096830.0,189182859744.0,195102981665.0,0.769716,-0.031293
13,05001,MEDELLÍN,2020,7.754609e+11,832944002309.0,195102981665.0,184741524129.0,0.754305,0.053108
14,05001,MEDELLÍN,2021,8.352043e+11,877875140472.0,184741524129.0,358392239976.0,0.785988,-0.939966


,codigo_departamento,departamento,codigo_municipio,municipio,año,R_ACT_FUT,R_ANT_FUT,L_CHIP,C_CHIP,C_CIERRE_CHIP,R_TOT_CUIPO,R_TOT_FUT,R_TOT,y_C,y_R
0,05,ANTIOQUIA,05001,MEDELLÍN,2007,NaN,NaN,274933035000.0,<NA>,159436192000.0,NaN,NaN,NaN,NaN,NaN
1,05,ANTIOQUIA,05001,MEDELLÍN,2008,NaN,NaN,366671863000.0,159436192000.0,170794237000.0,NaN,NaN,NaN,NaN,-0.071239
2,05,ANTIOQUIA,05001,MEDELLÍN,2009,NaN,NaN,366410885000.0,170794237000.0,204772075000.0,NaN,NaN,NaN,NaN,-0.198940
3,05,ANTIOQUIA,05001,MEDELLÍN,2010,NaN,NaN,388197844000.0,204772075000.0,229313294000.0,NaN,NaN,NaN,NaN,-0.119847
4,05,ANTIOQUIA,05001,MEDELLÍN,2011,NaN,NaN,426247886000.0,229313294000.0,243456864000.0,NaN,NaN,NaN,NaN,-0.061678
5,05,ANTIOQUIA,05001,MEDELLÍN,2012,3.760985e+08,4.791869e+07,455349723000.0,243456864000.0,272182803000.0,NaN,4.240172e+11,4.240172e+11,0.606773,-0.117992
6,05,ANTIOQUIA,05001,MEDELLÍN,2013,4.576889e+08,8.247165e+07,561463610000.0,272182803000.0,281461808000.0,NaN,5.401606e+11,5.401606e+11,0.647949,-0.034091
7,05,ANTIOQUIA,05001,MEDELLÍN,2014,4.935976e+08,6.027679e+07,600316238000.0,281461808000.0,318114186000.0,NaN,5.538744e+11,5.538744e+11,0.628134,-0.130221
8,05,ANTIOQUIA,05001,MEDELLÍN,2015,5.226282e+08,7.616222e+07,626565218000.0,318114186000.0,337406053000.0,NaN,5.987905e+11,5.987905e+11,0.633856,-0.060644
9,05,ANTIOQUIA,05001,MEDELLÍN,2016,5.517421e+08,6.199670e+07,660356013000.0,337406053000.0,374452020000.0,NaN,6.137388e+11,6.137388e+11,0.615115,-0.109796


In [9]:
# ============================================================
# BLOQUE FINAL. EXPORTAR PANEL INICIAL
# Periodo: 2007-2025
# ============================================================


# ------------------------------------------------------------
# 1. DEFINIR Y CREAR LA CARPETA DE SALIDA
# ------------------------------------------------------------

RUTA_PANEL = RUTA_DATOS / "panel"

RUTA_PANEL.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. DEFINIR EL NOMBRE DEL ARCHIVO
# ------------------------------------------------------------

archivo_panel = (
    RUTA_PANEL / "Panel_Inicial_2007_2025.xlsx"
)


# ------------------------------------------------------------
# 3. EXPORTAR EL PANEL A EXCEL
# ------------------------------------------------------------

panel.to_excel(
    archivo_panel,
    index=False
)


# ------------------------------------------------------------
# 4. CONFIRMAR LA EXPORTACIÓN
# ------------------------------------------------------------

print("Panel inicial guardado correctamente.")
print("Archivo:", archivo_panel)
print("Observaciones:", len(panel))
print("Variables:", len(panel.columns))

Panel inicial guardado correctamente.
Archivo: /content/drive/MyDrive/Datos/panel/Panel_Inicial_2007_2025.xlsx
Observaciones: 20957
Variables: 15
